# Deep Learning Based Crop Yield Prediction System
**Predicting Agricultural Crop Yield Using Environmental and Agricultural Factors**

This notebook walks through the same steps as `train.py`, one small cell at a time, so every step can be explained in a viva.
It does **not** overwrite the trained model in `models/` (the saved model comes from `python train.py`).

Pipeline: `Dataset -> Clean -> Train/Val/Test split -> Fit preprocessing on TRAIN only -> Dense neural network -> Evaluate on TEST`

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
keras.utils.set_random_seed(SEED)     # same random numbers every run
print("TensorFlow", tf.__version__, "| NumPy", np.__version__)

## 1. Load the dataset
Source: Kaggle *Crop Yield Prediction Dataset* (FAO yield and pesticide data + World Bank rainfall and temperature).
The yield column `hg/ha_yield` is in **hectograms per hectare**. 1 tonne = 10,000 hg, so `t/ha = hg/ha / 10000`.

In [ ]:
df = pd.read_csv("../dataset/crop_yield.csv")
df = df.drop(columns=["Unnamed: 0"])            # just the row index
df = df.rename(columns={
    "Area": "country", "Item": "crop", "Year": "year",
    "average_rain_fall_mm_per_year": "rainfall", "pesticides_tonnes": "pesticides",
    "avg_temp": "temperature", "hg/ha_yield": "yield_hg_ha"})
print(df.shape)
df.head()

## 2. Clean the data
Check data types, missing values and duplicates.

In [ ]:
print(df.dtypes, "\n")
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
df = df.dropna().drop_duplicates().reset_index(drop=True)
print("Rows after cleaning:", len(df))
df.describe().round(2)

## 3. Look at the data
Yield differs a lot between crops, which is why *crop* is an important input.

In [ ]:
df["yield_t_ha"] = df["yield_hg_ha"] / 10_000
order = df.groupby("crop")["yield_t_ha"].median().sort_values().index
df.boxplot(column="yield_t_ha", by="crop", figsize=(9, 4), rot=30, grid=False)
plt.suptitle(""); plt.title("Yield by crop"); plt.ylabel("Yield (t/ha)"); plt.show()

## 4. Split first, then fit preprocessing (no data leakage)
80% train+validation, 20% test. The scaler and encoder are fitted on the **training rows only**.

In [ ]:
features = ["country", "crop", "year", "rainfall", "pesticides", "temperature"]
X, y = df[features], df["yield_hg_ha"]

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.15, random_state=SEED)
print("train", len(X_train), "| validation", len(X_val), "| test", len(X_test))

In [ ]:
preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["country", "crop"]),
    ("scaled_numbers", StandardScaler(), ["year", "rainfall", "temperature"]),
    ("log_pesticides", Pipeline([("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]), ["pesticides"]),
])

Xt_train = preprocessor.fit_transform(X_train)   # fit ONLY on training data
Xt_val   = preprocessor.transform(X_val)
Xt_test  = preprocessor.transform(X_test)
print("Input features for the network:", Xt_train.shape[1])

# standardise the target too (mean/std from TRAIN only); converted back after prediction
y_mean, y_std = y_train.mean(), y_train.std()
z_train = ((y_train - y_mean) / y_std).values
z_val   = ((y_val - y_mean) / y_std).values

## 5. Build the neural network
`Input -> Dense(128, ReLU) -> Dropout(0.2) -> Dense(64, ReLU) -> Dropout(0.2) -> Dense(32, ReLU) -> Dense(1, linear)`

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(Xt_train.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="linear"),
])
model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mean_squared_error", metrics=["mean_absolute_error"])
model.summary()

## 6. Train
EarlyStopping stops training when the validation loss stops improving and keeps the best weights.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)
history = model.fit(Xt_train, z_train, validation_data=(Xt_val, z_val),
                    epochs=300, batch_size=64, callbacks=[early_stop], verbose=0)
print("Stopped after", len(history.history["loss"]), "epochs")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.yscale("log"); plt.xlabel("Epoch"); plt.ylabel("Loss (MSE, standardised yield)")
plt.title("Training vs validation loss"); plt.legend(); plt.show()

## 7. Evaluate on the test set
The network output is converted back: standardised -> hg/ha -> **tonnes/hectare**.

In [ ]:
pred_hg = model.predict(Xt_test, verbose=0).ravel() * y_std + y_mean
actual_t, pred_t = y_test.values / 10_000, pred_hg / 10_000

print(f"MAE  : {mean_absolute_error(actual_t, pred_t):.3f} t/ha")
print(f"MSE  : {mean_squared_error(actual_t, pred_t):.3f}")
print(f"RMSE : {np.sqrt(mean_squared_error(actual_t, pred_t)):.3f} t/ha")
print(f"R2   : {r2_score(actual_t, pred_t):.4f}")

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(actual_t, pred_t, s=6, alpha=0.3)
top = max(actual_t.max(), pred_t.max())
plt.plot([0, top], [0, top], color="orange")
plt.xlabel("Actual yield (t/ha)"); plt.ylabel("Predicted yield (t/ha)"); plt.title("Actual vs predicted (test set)")
plt.show()

## 8. Predict for one input
This is exactly what the Flask app does for every request.

In [ ]:
sample = pd.DataFrame([{"country": "India", "crop": "Potatoes", "year": 2013,
                        "rainfall": 1083, "pesticides": 45620, "temperature": 26.71}])
z = model.predict(preprocessor.transform(sample), verbose=0)[0, 0]
tonnes_per_hectare = (z * y_std + y_mean) / 10_000
print(f"Predicted Crop Yield: {tonnes_per_hectare:.2f} tonnes/hectare")